In [4]:
from datasets import load_dataset

ds = load_dataset("Salesforce/wikitext", "wikitext-103-raw-v1")

c:\Users\NITRO\miniconda3\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
c:\Users\NITRO\miniconda3\Lib\site-packages\huggingface_hub\file_download.py:129: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\NITRO\.cache\huggingface\hub\datasets--Salesforce--wikitext. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this

In [11]:
ds

DatasetDict({
    test: Dataset({
        features: ['text'],
        num_rows: 4358
    })
    train: Dataset({
        features: ['text'],
        num_rows: 1801350
    })
    validation: Dataset({
        features: ['text'],
        num_rows: 3760
    })
})

In [12]:
small_train = ds["train"].select(range(50000))
small_valid = ds["validation"]
small_test  = ds["test"]

In [13]:
small_train

Dataset({
    features: ['text'],
    num_rows: 50000
})

In [14]:
for i in range(5):
    print(repr(small_train[i]["text"]))

''
' = Valkyria Chronicles III = \n'
''
' Senjō no Valkyria 3 : Unrecorded Chronicles ( Japanese : 戦場のヴァルキュリア3 , lit . Valkyria of the Battlefield 3 ) , commonly referred to as Valkyria Chronicles III outside Japan , is a tactical role @-@ playing video game developed by Sega and Media.Vision for the PlayStation Portable . Released in January 2011 in Japan , it is the third game in the Valkyria series . Employing the same fusion of tactical and real @-@ time gameplay as its predecessors , the story runs parallel to the first game and follows the " Nameless " , a penal military unit serving the nation of Gallia during the Second Europan War who perform secret black operations and are pitted against the Imperial unit " Calamaty Raven " . \n'
" The game began development in 2010 , carrying over a large portion of the work done on Valkyria Chronicles II . While it retained the standard features of the series , it also underwent multiple adjustments , such as making the game more forgiving 

In [15]:
train_texts = [x["text"] for x in small_train if x["text"].strip()]
valid_texts = [x["text"] for x in small_valid if x["text"].strip()]
test_texts  = [x["text"] for x in small_test if x["text"].strip()]

Tokenize 


In [16]:
import re
from collections import Counter

def tokenize(text):
    return re.findall(r"\b\w+\b|[^\w\s]", text.lower())

train_tokens = []
for txt in train_texts:
    train_tokens.extend(tokenize(txt))

la creation d un dictionnaire

In [24]:
counter = Counter(train_tokens)
vocab = ["<pad>", "<unk>"] + [w for w, c in counter.items() if c >= 2]

word2idx = {w: i for i, w in enumerate(vocab)}
idx2word = {i: w for w, i in word2idx.items()}

def encode(tokens):
    return [word2idx.get(t, word2idx["<unk>"]) for t in tokens]

train_ids = encode(train_tokens)

In [25]:
seq_len = 5

X, Y = [], []
for i in range(len(train_ids) - seq_len):
    X.append(train_ids[i:i+seq_len])
    Y.append(train_ids[i+1:i+seq_len+1])

In [27]:
X

[[2, 3, 4, 5, 2],
 [3, 4, 5, 2, 6],
 [4, 5, 2, 6, 7],
 [5, 2, 6, 7, 3],
 [2, 6, 7, 3, 8],
 [6, 7, 3, 8, 9],
 [7, 3, 8, 9, 10],
 [3, 8, 9, 10, 4],
 [8, 9, 10, 4, 11],
 [9, 10, 4, 11, 12],
 [10, 4, 11, 12, 9],
 [4, 11, 12, 9, 13],
 [11, 12, 9, 13, 14],
 [12, 9, 13, 14, 15],
 [9, 13, 14, 15, 16],
 [13, 14, 15, 16, 3],
 [14, 15, 16, 3, 17],
 [15, 16, 3, 17, 18],
 [16, 3, 17, 18, 19],
 [3, 17, 18, 19, 8],
 [17, 18, 19, 8, 20],
 [18, 19, 8, 20, 14],
 [19, 8, 20, 14, 21],
 [8, 20, 14, 21, 22],
 [20, 14, 21, 22, 23],
 [14, 21, 22, 23, 24],
 [21, 22, 23, 24, 3],
 [22, 23, 24, 3, 4],
 [23, 24, 3, 4, 5],
 [24, 3, 4, 5, 25],
 [3, 4, 5, 25, 26],
 [4, 5, 25, 26, 14],
 [5, 25, 26, 14, 27],
 [25, 26, 14, 27, 28],
 [26, 14, 27, 28, 29],
 [14, 27, 28, 29, 30],
 [27, 28, 29, 30, 31],
 [28, 29, 30, 31, 32],
 [29, 30, 31, 32, 31],
 [30, 31, 32, 31, 33],
 [31, 32, 31, 33, 34],
 [32, 31, 33, 34, 35],
 [31, 33, 34, 35, 36],
 [33, 34, 35, 36, 37],
 [34, 35, 36, 37, 38],
 [35, 36, 37, 38, 39],
 [36, 37, 38, 39,

In [33]:
import numpy as np

X = np.array(X)
print(X.shape)

(2872040, 5)


In [ ]:
import math
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import TensorDataset, DataLoader

# -----------------------------
# 1) TES DONNEES
# -----------------------------
data = [
    [2, 3, 4, 5, 2],
    [3, 4, 5, 2, 6],
    [4, 5, 2, 6, 7],
    [5, 2, 6, 7, 3],
    [2, 6, 7, 3, 8],
    [6, 7, 3, 8, 9],
    [7, 3, 8, 9, 10],
    [3, 8, 9, 10, 4],
    [8, 9, 10, 4, 11],
    [9, 10, 4, 11, 12],
    [10, 4, 11, 12, 9],
    [4, 11, 12, 9, 13],
    [11, 12, 9, 13, 14],
    [12, 9, 13, 14, 15],
    [9, 13, 14, 15, 16],
    [13, 14, 15, 16, 3],
    [14, 15, 16, 3, 17],
    [15, 16, 3, 17, 18],
    [16, 3, 17, 18, 19],
    [3, 17, 18, 19, 8],
    [17, 18, 19, 8, 20],
    [18, 19, 8, 20, 14],
    [19, 8, 20, 14, 21],
    [8, 20, 14, 21, 22],
    [20, 14, 21, 22, 23],
    [409, 120, 18, 410, 39],
    [120, 18, 410, 39, 1],
    [18, 410, 39, 1, 411],
    [410, 39, 1, 411, 22],
]

# X = séquence courante
# Y = séquence suivante (donc token cible à chaque pas)
X_data = torch.tensor(data[:-1], dtype=torch.long)
Y_data = torch.tensor(data[1:], dtype=torch.long)

vocab_size = int(torch.tensor(data).max().item()) + 1
seq_len = X_data.shape[1]

dataset = TensorDataset(X_data, Y_data)
loader = DataLoader(dataset, batch_size=8, shuffle=True)

# -----------------------------
# 2) RNN FROM SCRATCH
# -----------------------------
class RNNFromScratch(nn.Module):
    def __init__(self, vocab_size, emb_dim, hidden_dim):
        super().__init__()
        self.vocab_size = vocab_size
        self.emb_dim = emb_dim
        self.hidden_dim = hidden_dim

        # Embedding appris
        self.embedding = nn.Parameter(torch.randn(vocab_size, emb_dim) * 0.01)

        # Paramètres du RNN
        self.Wxh = nn.Parameter(torch.randn(emb_dim, hidden_dim) / math.sqrt(emb_dim))
        self.Whh = nn.Parameter(torch.randn(hidden_dim, hidden_dim) / math.sqrt(hidden_dim))
        self.bh = nn.Parameter(torch.zeros(hidden_dim))

        # Tête de sortie
        self.Why = nn.Parameter(torch.randn(hidden_dim, vocab_size) / math.sqrt(hidden_dim))
        self.by = nn.Parameter(torch.zeros(vocab_size))

    def forward(self, x, h0=None):
        # x: [batch, 5]
        batch_size, T = x.shape

        if h0 is None:
            h = torch.zeros(batch_size, self.hidden_dim, device=x.device)
        else:
            h = h0

        logits_all = []
        hidden_all = []

        for t in range(T):
            x_t = x[:, t]                    # [batch]
            emb_t = self.embedding[x_t]      # [batch, emb_dim]

            h = torch.tanh(emb_t @ self.Wxh + h @ self.Whh + self.bh)   # [batch, hidden_dim]
            logits_t = h @ self.Why + self.by                           # [batch, vocab_size]

            logits_all.append(logits_t.unsqueeze(1))   # [batch, 1, vocab]
            hidden_all.append(h.unsqueeze(1))          # [batch, 1, hidden]

        logits_all = torch.cat(logits_all, dim=1)      # [batch, 5, vocab_size]
        hidden_all = torch.cat(hidden_all, dim=1)      # [batch, 5, hidden_dim]

        return logits_all, hidden_all, h

# -----------------------------
# 3) INITIALISATION
# -----------------------------
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = RNNFromScratch(
    vocab_size=vocab_size,
    emb_dim=32,
    hidden_dim=64
).to(device)

optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

# -----------------------------
# 4) ENTRAINEMENT
# -----------------------------
epochs = 100

for epoch in range(epochs):
    model.train()
    total_loss = 0.0
    total_acc = 0.0
    n_batches = 0

    for xb, yb in loader:
        xb = xb.to(device)
        yb = yb.to(device)

        optimizer.zero_grad()

        logits, hidden_states, h_last = model(xb)
        loss = F.cross_entropy(logits.reshape(-1, vocab_size), yb.reshape(-1))

        loss.backward()
        optimizer.step()

        preds = logits.argmax(dim=-1)
        acc = (preds == yb).float().mean().item()

        total_loss += loss.item()
        total_acc += acc
        n_batches += 1

    if (epoch + 1) % 10 == 0:
        print(f"Epoch {epoch+1:03d} | loss={total_loss/n_batches:.4f} | acc={total_acc/n_batches:.4f}")


sample = torch.tensor([[2, 3, 4, 5, 2]], dtype=torch.long).to(device)

with torch.no_grad():
    logits, hidden_states, h_last = model(sample)
    pred_seq = logits.argmax(dim=-1)

print("Input         :", sample.cpu().tolist()[0])
print("Prediction    :", pred_seq.cpu().tolist()[0])

Epoch 010 | loss=4.5896 | acc=0.2062
Epoch 020 | loss=3.3317 | acc=0.3063
Epoch 030 | loss=2.4726 | acc=0.5438
Epoch 040 | loss=1.6992 | acc=0.7750
Epoch 050 | loss=1.0744 | acc=0.9062
Epoch 060 | loss=0.7286 | acc=0.9375
Epoch 070 | loss=0.5374 | acc=0.9375
Epoch 080 | loss=0.3955 | acc=0.9375
Epoch 090 | loss=0.3051 | acc=0.9563
Epoch 100 | loss=0.2500 | acc=0.9500
Input         : [2, 3, 4, 5, 2]
Prediction    : [6, 4, 5, 2, 6]


In [37]:
import torch
import torch.nn.functional as F

def evaluate_model(model, dataloader, vocab_size, device):
    model.eval()

    total_loss = 0.0
    total_correct = 0
    total_tokens = 0

    all_preds = []
    all_targets = []

    with torch.no_grad():
        for xb, yb in dataloader:
            xb = xb.to(device)
            yb = yb.to(device)

            logits, _, _ = model(xb)   # [batch, seq_len, vocab_size]

            loss = F.cross_entropy(
                logits.reshape(-1, vocab_size),
                yb.reshape(-1),
                reduction="sum"
            )

            preds = logits.argmax(dim=-1)   # [batch, seq_len]

            total_loss += loss.item()
            total_correct += (preds == yb).sum().item()
            total_tokens += yb.numel()

            all_preds.append(preds.cpu())
            all_targets.append(yb.cpu())

    avg_loss = total_loss / total_tokens
    accuracy = total_correct / total_tokens
    perplexity = torch.exp(torch.tensor(avg_loss)).item()

    all_preds = torch.cat(all_preds, dim=0)
    all_targets = torch.cat(all_targets, dim=0)

    return {
        "loss": avg_loss,
        "accuracy": accuracy,
        "perplexity": perplexity,
        "predictions": all_preds,
        "targets": all_targets
    }

In [38]:
results = evaluate_model(model, loader, vocab_size, device)

print("Loss       :", results["loss"])
print("Accuracy   :", results["accuracy"])
print("Perplexity :", results["perplexity"])

Loss       : 0.24531989097595214
Accuracy   : 0.95
Perplexity : 1.2780300378799438
